# P10: Bi-LSTM bilan nomlangan obyektlarni tanib olish (NER)
**Mavzu:** IOB2 teglash, Embedding + Bi-LSTM, per-token teg, entity ajratish, F1
**Kun:** 11-kun amaliyoti — 2-iyul 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L10 — Nomlangan obyektlarni tanib olish (`d10_ner.pdf`)
**Kapstone modul:** m10 `NERTagger`
**Ma'lumot:** WikiANN uz (~200 jumlat). Offline: `d11_checkpoints/uz_ner_mini.txt` (original IOB2).

## Bugungi maqsadlar (ma'ruza [C] dan)
1. IOB2 teglashni qo'lda tekshiramiz ("Malika Toshkentda ishlaydi ." → [B-PER, B-LOC, O, O]).
2. Embedding + Bi-LSTM bilan NER modelini quramiz va o'qitamiz.
3. Har token uchun teg bashorat qilamiz; entitylarni ajratamiz.
4. F1 bilan baholaymiz (kam ma'lumot → past F1, halol).

## Vaqt taqsimoti
| Bo'lim | Vaqt | Mazmun |
|---|---|---|
| §1 Muhit | 4 daq | seedlar, OFFLINE_FALLBACK, HAS_TORCH, HAS_SEQEVAL |
| §2 Yaxlit natija | 8 daq | tayyor NER — entities() demo |
| §3 Periferiya (PRIMM) | 17 daq | IOB2 yuklash + teg/indeks kodlash; padding/masking loop |
| §4 Yadro | 35 daq | IOB2 gold (qulflangan), Bi-LSTM NER qurish/o'qitish, predict/entities, F1 |
| §5 Loyihaga ulash | 13 daq | m10 moduli, import test, git |
| §6 Tadqiqot + yakun | 7 daq | -da/-dan suffiks; nega kam ma'lumotda F1 past |

## 1. Muhit tekshiruvi
Seedlar, offline bayroq, `torch` va `seqeval` bayroqlari. GPU shart emas (CPU).

In [ ]:
# Kaggle: torch + seqeval oldindan o'rnatilgan (GPU bo'lsa tezlashadi, talab emas)
import os, sys, random
import numpy as np

random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True

try:
    import torch
    torch.manual_seed(42)
    HAS_TORCH = True
    print("torch mavjud:", torch.__version__)
except ImportError:
    HAS_TORCH = False
    print("torch yo'q — reservoir Bi-LSTM + LogReg (offline).")

try:
    import seqeval
    HAS_SEQEVAL = True
except ImportError:
    HAS_SEQEVAL = False
    print("seqeval yo'q — token-darajali F1 (sklearn) ishlatiladi.")

MODULES = os.path.join("..", "..", "capstone", "modules")
if not os.path.isdir(MODULES):
    MODULES = os.path.join("capstone", "modules")
sys.path.insert(0, MODULES)
print("numpy:", np.__version__, "| OFFLINE_FALLBACK =", OFFLINE_FALLBACK)


## 2. Yaxlit natija (avval manzil)
Tayyor `NERTagger` IOB2 korpusda o'qitiladi va o'zbek matnidan shaxs/joy/tashkilotlarni
ajratadi. (Kichik korpus → F1 past, lekin tizim ishlaydi.)

In [ ]:
from m10_ner_tagger import NERTagger

CONLL = os.path.join("..", "..", "course", "practices", "d11_checkpoints", "uz_ner_mini.txt")
if not os.path.exists(CONLL):
    CONLL = os.path.join("course", "practices", "d11_checkpoints", "uz_ner_mini.txt")

def load_conll(path):
    """CoNLL (token<TAB>teg, bo'sh qator = jumla chegarasi) -> [[(token,teg),...],...]."""
    sents, cur = [], []
    for line in open(path, encoding="utf-8"):
        line = line.rstrip("\n")
        if not line.strip():
            if cur: sents.append(cur); cur = []
            continue
        tok, tag = line.split("\t")
        cur.append((tok, tag))
    if cur: sents.append(cur)
    return sents

sents = load_conll(CONLL)
print("Jumlalar:", len(sents), "| tokenlar:", sum(len(s) for s in sents))

ner_demo = NERTagger(); ner_demo.fit(sents, epochs=15)
print("Entitylar:", ner_demo.entities("Akmal Samarqandda yashaydi ."))


## 3. Tayyor kod bloki — IOB2 yuklash va kodlash (PRIMM)
Bu bo'lim periferiya. NER ma'lumoti — IOB2 teglangan jumlalar. Teglarni indekslarga
o'giramiz va (torch yo'lida) padding/masking qo'llaymiz.

**Bashorat qiling:** korpusda nechta xil teg bor? `O` teg ham indeks oladimi?

In [ ]:
# PERIFERIYA — to'liq beriladi. teg/indeks kodlash.
tags = sorted({tag for s in sents for _, tag in s})
if "O" in tags:
    tags.remove("O"); tags = ["O"] + tags         # O ni 0-indeksga
t2i = {t: i for i, t in enumerate(tags)}
words = sorted({tok for s in sents for tok, _ in s})
w2i = {w: i + 1 for i, w in enumerate(words)}      # 0 = PAD/UNK

print("Teglar:", tags)
print("Lug'at hajmi:", len(w2i), "| teglar soni:", len(tags))
# bitta jumlani kodlash
demo = sents[0]
print("Kodlangan:", [(w2i[t], t2i[g]) for t, g in demo])


**Tekshiring:**
1. Nega `O` ni 0-indeksga qo'ydik? (Eng ko'p uchraydigan teg.)
2. `B-LOC` va `I-LOC` alohida indekslarmi? Nega kerak? (entity chegarasi.)
3. Padding nima uchun kerak (torch yo'lida)? Masking nimani qiladi?

**O'zgartiring:** o'z jumlangizni IOB2 da teglab, kodlanishini ko'ring.

In [ ]:
# PERIFERIYA — padding va masking sxemasi (torch yo'lida; izohli).
if HAS_TORCH:
    seqs = [[w2i[t] for t, _ in s] for s in sents]
    T = max(len(s) for s in seqs)
    padded = [s + [0] * (T - len(s)) for s in seqs]    # 0 = PAD
    mask = [[1] * len(s) + [0] * (T - len(s)) for s in seqs]  # 1=haqiqiy, 0=pad
    print("Eng uzun jumla:", T, "token | padding 0 bilan; teg loss'da -100 e'tiborsiz.")
else:
    print("Offline: reservoir har jumlani alohida qayta ishlaydi (padding shart emas).")


### Checkpoint A
Orqada qolsangiz — IOB2 korpusni qaytadan yuklaydi.

In [ ]:
if OFFLINE_FALLBACK or "sents" not in dir():
    sents = load_conll(CONLL)
print("Checkpoint: korpus tayyor —", len(sents), "jumla")


## 4. Asosiy mavzu — NER ni qadam-baqadam (so'nuvchi tayanch)
**Namuna → Birgalikda → Mustaqil.** Avval ma'ruzaning IOB2 misolini takrorlaymiz.

### 4A. Namuna (men qilaman): IOB2 gold teglash
Ma'ruza L10 [I1]: "Malika Toshkentda ishlaydi ." ni **qo'lda** (gold) teglaymiz. Bu model
bashorati emas — IOB2 sxemasini va kuzatiluvchanlikni tekshiradi (toza-python).

In [ ]:
# IOB2 gold teglash (qo'lda) — L10 [I1]
gap = "Malika Toshkentda ishlaydi ."
tokens = gap.split()                          # [Malika, Toshkentda, ishlaydi, .]
gold_tags = ["B-PER", "B-LOC", "O", "O"]      # Toshkentda=B-LOC ("-da" chegarani buzmaydi)
print(list(zip(tokens, gold_tags)))

assert gold_tags == ["B-PER", "B-LOC", "O", "O"]   # Ma'ruza L10 [I1]-slayd bilan solishtiring
print("To'g'ri! IOB2 gold teglar = [B-PER, B-LOC, O, O]")


### 4B. Birgalikda (biz qilamiz): Bi-LSTM NER modelini o'qitamiz
`NERTagger` ichida Embedding + Bi-LSTM (Kaggle) yoki reservoir (offline). Har token uchun
IOB2 teg bashorat qiladi. CPU uchun kichik o'lcham va kam epoch.

In [ ]:
ner = NERTagger()
# === SIZNING KODINGIZ (taxminan 2 qator) ===
# 1) ner ni sents (IOB2 jumlalar) ustida o'qiting: epochs=15
# 2) "Malika Toshkentda ishlaydi ." ni predict qiling
ner.fit(...)
pred = ...
print(pred)


In [ ]:
assert len(ner._tags) > 0 and len(ner._w2i) > 0, "Lug'at/teglar bo'sh — fit() ni tekshiring."
assert isinstance(pred, list) and all(
    isinstance(p, tuple) and len(p) == 2 and isinstance(p[0], str) and isinstance(p[1], str)
    for p in pred), "predict() (token, teg) juftlar ro'yxatini qaytarishi kerak."
IOB2 = {"O","B-PER","I-PER","B-LOC","I-LOC","B-ORG","I-ORG"}
assert all(tag in IOB2 for _, tag in pred), "Teglar IOB2 to'plamidan bo'lishi kerak."
print("Ajoyib! Model o'qidi — teglar:", ner._tags)


### 4C. Birgalikda (biz qilamiz): entitylarni ajratamiz
`entities()` IOB2 teglardan to'liq entitylarni yig'adi: matn, tur (label), char chegaralari.

In [ ]:
# === SIZNING KODINGIZ (taxminan 1 qator) ===
# "Akmal Samarqandda yashaydi ." dan entitylarni ajrating
ents = ...
for e in ents:
    print(e)


In [ ]:
assert isinstance(ents, list), "entities() ro'yxat qaytarishi kerak."
assert all(isinstance(e, dict) and {"text","label","start","end"} <= set(e.keys()) for e in ents), \
    "Har entity {text, label, start, end} kalitlariga ega dict bo'lishi kerak."
print("To'g'ri! entities() {text,label,start,end} dict ro'yxatini qaytardi.")


### 4D. Mustaqil (siz qilasiz): F1 baholash va saqlash
Token-darajali F1 ni hisoblang (seqeval bo'lsa entity-darajali). Kam ma'lumot →
**F1 past bo'ladi** — bu o'zbek kabi kam-resursli tillar uchun halol holat. Modelni
saqlang/yuklang. Tayanch yo'q.

In [ ]:
# === SIZNING KODINGIZ (taxminan 10 qator) ===
# 1) barcha jumlalar uchun gold va predict teglarni yig'ing
# 2) token-darajali macro-F1 ni sklearn bilan hisoblang (zero_division=0)
# 3) assert: 0 <= f1 <= 1
# 4) modelni saqlang/yuklang; assert: bashorat mos keladi
from sklearn.metrics import f1_score
import tempfile
...


## 5. Loyihaga ulash — m10 `NERTagger`
Bugungi ish `capstone/modules/m10_ner_tagger.py` da (shartnoma `capstone/contracts.py`).
Torch-ixtiyoriy + seqeval-ixtiyoriy; **save/load bor** (m15 agent va Day 16 pipeline ishlatadi).

In [ ]:
from m10_ner_tagger import NERTagger

m = NERTagger()
for metod in ["fit", "predict", "entities", "save", "load"]:
    assert callable(getattr(m, metod, None)), f"m10.{metod} mavjud emas!"
m.fit(sents, epochs=3)
out = m.predict("Malika Toshkentda ishlaydi .")
assert isinstance(out, list) and isinstance(out[0], tuple), "predict() list[tuple] qaytaradi."
print("m10 NERTagger shartnomaga mos: fit / predict / entities / save / load")


```bash
git add capstone/modules/m10_ner_tagger.py
git commit -m "P10: m10 NERTagger — Bi-LSTM IOB2 NER"
git push
```

## 6. Tadqiqot savoli + yakun
**Mini-tajriba:** "Toshkentda" (B-LOC) va "Toshkentdan" (B-LOC) — qo'shimchali joy nomlari.
Modelingiz ularni qanday tegladi? Suffiks entity turini o'zgartirdimi?

In [ ]:
for matn in ["Aziz Toshkentda yashaydi .", "Aziz Toshkentdan keldi ."]:
    print(matn, "->", ner.predict(matn))
print("Eslatma: '-da'/'-dan' suffiks joy nomini o'zgartirmaydi — token butun B-LOC bo'lishi kerak.")


**Savol (o'ylab ko'ring):** WikiANN uz'da atigi ~200 jumlat bor — shuning uchun F1 past.
Bu o'zbek kabi kam-resursli tillarda keng tarqalgan muammo. Uni qanday yengish mumkin?
(Maslahat: ko'p tilli BERT, transfer learning — 4-hafta.)

**Chiqish chiptasi:** _Bugun eng tushunarsiz qolgan narsa nima?_